# Data Cleaning — Product Recommendation System
Cleans the raw e-commerce behavioural event dataset before it enters the feature engineering stage.

**Handles:**
- Missing/null values in critical fields
- Exact duplicate rows, including duplicates across processing chunks
- Malformed / bad entries (invalid event types, non-positive price/IDs)
- Renaming columns to match the project shared event schema
- Saving the cleaned dataset as a single CSV file
- Chunked processing so the full raw dataset is not loaded into RAM at once

**Set your paths in the cell below, then run the notebook top to bottom.**


## 0. Paths — edit these, nothing else in the notebook needs to change

In [5]:
INPUT_PATH = r"D:\Product-Recommendation-System\data\dataset\2019-Oct.csv"
OUTPUT_PATH = r"D:\Product-Recommendation-System\data\dataset\cleaned\cleaned_events.csv"

print("Input :", INPUT_PATH)
print("Output:", OUTPUT_PATH)


Input : D:\Product-Recommendation-System\data\dataset\2019-Oct.csv
Output: D:\Product-Recommendation-System\data\dataset\cleaned\cleaned_events.csv


## 1. Imports & Logging Setup

In [2]:
import hashlib
import logging
import os
import sqlite3

import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

logger = logging.getLogger(__name__)


## 2. Config — required fields and valid event types

In [3]:
REQUIRED_COLUMNS = [
    "event_time",
    "event_type",
    "product_id",
    "user_id"
]

VALID_EVENT_TYPES = {
    "view",
    "cart",
    "purchase",
    "remove_from_cart"
}

# Keep this reasonably small so the local machine does not run out of RAM.
CHUNK_SIZE = 100_000


## 3. Step 1 — Load raw data in chunks


In [4]:
def load_data(path: str):
    """Read the raw CSV incrementally instead of loading the whole file into RAM."""
    logger.info(f"Loading raw data from {path}")
    return pd.read_csv(
        path,
        chunksize=CHUNK_SIZE,
        low_memory=False
    )


## 4. Step 2 — Drop missing values in critical fields

In [6]:
def drop_missing_critical_fields(df: pd.DataFrame) -> pd.DataFrame:
    """Remove rows where critical fields are missing."""
    before = len(df)
    df = df.dropna(subset=REQUIRED_COLUMNS)
    logger.info(f"Dropped {before - len(df):,} rows with missing critical fields")
    return df

## 5. Step 3 — Drop exact duplicate rows

In [7]:
def drop_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    """Remove exact duplicate rows."""
    before = len(df)
    df = df.drop_duplicates()
    logger.info(f"Dropped {before - len(df):,} duplicate rows")
    return df

## 6. Step 4 — Remove bad / malformed entries

In [8]:
def remove_bad_entries(df: pd.DataFrame) -> pd.DataFrame:
    """Remove invalid event types, IDs, and prices."""
    before = len(df)

    # Validate event types
    df = df[df["event_type"].isin(VALID_EVENT_TYPES)].copy()

    # Convert numeric columns safely
    df["product_id"] = pd.to_numeric(df["product_id"], errors="coerce")
    df["user_id"] = pd.to_numeric(df["user_id"], errors="coerce")

    # Validate product and user IDs
    df = df[
        (df["product_id"] > 0) &
        (df["user_id"] > 0)
    ]

    # Validate price if it exists
    if "price" in df.columns:
        df["price"] = pd.to_numeric(df["price"], errors="coerce")
        df = df[df["price"] > 0]

    logger.info(f"Dropped {before - len(df):,} bad/malformed rows")
    return df

## 7. Step 5 — Standardize schema

In [9]:
def standardize_schema(df: pd.DataFrame) -> pd.DataFrame:
    """Rename columns and standardize timestamps."""
    df = df.rename(
        columns={
            "product_id": "item_id",
            "user_session": "session_id",
            "event_time": "timestamp"
        }
    )

    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

    before = len(df)
    df = df.dropna(subset=["timestamp"])
    logger.info(f"Dropped {before - len(df):,} rows with unparseable timestamps")

    return df

## 8. Step 6 — Save cleaned dataset (CSV only)


In [10]:
def save_data(df: pd.DataFrame, output_path: str, first_chunk: bool) -> None:
    """Append a cleaned chunk to the output CSV."""
    output_directory = os.path.dirname(output_path)
    if output_directory:
        os.makedirs(output_directory, exist_ok=True)

    df.to_csv(
        output_path,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False
    )


## 9. Main preprocessing pipeline


In [11]:
def preprocess(input_path: str, output_path: str) -> pd.DataFrame:
    """Run the complete memory-efficient preprocessing pipeline."""
    output_directory = os.path.dirname(output_path)
    if output_directory:
        os.makedirs(output_directory, exist_ok=True)

    if os.path.exists(output_path):
        os.remove(output_path)

    # Temporary SQLite database lets us deduplicate across chunks without keeping
    # the entire dataset in RAM. It is deleted automatically when processing ends.
    temp_db = output_path + ".dedup.sqlite"
    if os.path.exists(temp_db):
        os.remove(temp_db)

    conn = sqlite3.connect(temp_db)
    conn.execute("PRAGMA journal_mode=OFF")
    conn.execute("PRAGMA synchronous=OFF")
    conn.execute("CREATE TABLE seen (row_hash BLOB PRIMARY KEY)")
    conn.commit()

    first_chunk = True
    total_before = 0
    total_after = 0
    total_duplicates = 0

    try:
        for chunk_number, df in enumerate(load_data(input_path), start=1):
            before = len(df)
            total_before += before

            df = drop_missing_critical_fields(df)
            df = drop_duplicates(df)
            df = remove_bad_entries(df)
            df = standardize_schema(df)

            if df.empty:
                logger.info(f"Chunk {chunk_number}: {before:,} → 0 rows")
                continue

            # Create a stable hash for each cleaned row and remove duplicates
            # that appeared in earlier chunks as well.
            row_hashes = pd.util.hash_pandas_object(
                df,
                index=False
            ).astype("uint64")

            unique_mask = []
            new_hashes = []
            for value in row_hashes.tolist():
                digest = hashlib.sha256(str(value).encode()).digest()
                exists = conn.execute(
                    "SELECT 1 FROM seen WHERE row_hash = ?",
                    (digest,)
                ).fetchone()
                if exists:
                    unique_mask.append(False)
                    total_duplicates += 1
                else:
                    unique_mask.append(True)
                    new_hashes.append((digest,))

            if new_hashes:
                conn.executemany(
                    "INSERT INTO seen(row_hash) VALUES (?)",
                    new_hashes
                )
                conn.commit()

            df = df.loc[unique_mask]
            total_after += len(df)

            if not df.empty:
                save_data(df, output_path, first_chunk)
                first_chunk = False

            logger.info(
                f"Chunk {chunk_number}: {before:,} input rows → "
                f"{len(df):,} output rows"
            )

            del df
            del row_hashes

    finally:
        conn.close()
        if os.path.exists(temp_db):
            os.remove(temp_db)

    logger.info("=" * 60)
    logger.info("PREPROCESSING COMPLETE")
    logger.info(f"Rows read:             {total_before:,}")
    logger.info(f"Rows written:          {total_after:,}")
    logger.info(f"Duplicates removed:   {total_duplicates:,}")
    logger.info(f"Total rows removed:   {total_before - total_after:,}")
    logger.info(f"Output: {output_path}")

    # Return only a small preview so we do not reload the entire output into RAM.
    if os.path.exists(output_path):
        return pd.read_csv(output_path, nrows=10)
    return pd.DataFrame()


## 10. Run it

The pipeline processes the file in chunks and writes the cleaned result directly to the requested local CSV path. A temporary SQLite file is used only during the run to track exact duplicate rows across chunks; it is deleted automatically when processing finishes.


In [12]:
cleaned_df = preprocess(INPUT_PATH, OUTPUT_PATH)


2026-09-12 18:30:03,296 | INFO | Loading raw data from D:\Product-Recommendation-System\data\dataset\2019-Oct.csv
2026-09-12 18:30:04,768 | INFO | Dropped 0 rows with missing critical fields
2026-09-12 18:30:05,155 | INFO | Dropped 17 duplicate rows
2026-09-12 18:30:05,298 | INFO | Dropped 119 bad/malformed rows
2026-09-12 18:30:05,783 | INFO | Dropped 0 rows with unparseable timestamps
2026-09-12 18:30:20,731 | INFO | Chunk 1: 100,000 input rows → 99,864 output rows
2026-09-12 18:30:21,785 | INFO | Dropped 0 rows with missing critical fields
2026-09-12 18:30:22,080 | INFO | Dropped 121 duplicate rows
2026-09-12 18:30:22,160 | INFO | Dropped 109 bad/malformed rows
2026-09-12 18:30:22,462 | INFO | Dropped 0 rows with unparseable timestamps
2026-09-12 18:30:43,283 | INFO | Chunk 2: 100,000 input rows → 99,770 output rows
2026-09-12 18:30:44,484 | INFO | Dropped 0 rows with missing critical fields
2026-09-12 18:30:44,707 | INFO | Dropped 27 duplicate rows
2026-09-12 18:30:44,809 | INFO | 

KeyboardInterrupt: 

## 11. Preview — top 10 rows of the cleaned dataset

In [ ]:
cleaned_df.head(10)
